# Interfaz para bases SQL usando SQLite3

## Instrucciones

## Iniciar programa

### Cargando paquetes necesarios

In [ ]:
!pip install -q gradio || echo "Ya está instalada"

In [ ]:
import os
import sqlite3
import pandas as pd
import gradio as gr
import urllib.request

### Preparando bases de datos

In [ ]:
!git clone https://github.com/zyntonyson/dbs_sql.git

In [ ]:
DB_DIR='dbs_sql/dbs'
DBS = {f.replace('.db','').title():os.path.join(DB_DIR,f) for f in os.listdir(DB_DIR) if f.endswith(".db")}


### Preparando Interfaz


In [ ]:
def run_query(db_path, query):
    if not query.strip():
        return pd.DataFrame({"Error": ["Debes ingresar una consulta SQL."]})
    try:
        with sqlite3.connect(db_path) as conn:
            df = pd.read_sql_query(query, conn)
        return df
    except Exception as e:
        return pd.DataFrame({"Error": [str(e)]})

In [ ]:
def make_interface():
    with gr.Blocks(title="SQLite Query Runner") as demo:
        gr.Markdown("## 🧮 Consultas SQL")

        with gr.Row():
            # Lado izquierdo: selección + query + botón
            with gr.Column(scale=1):
                db_select = gr.Dropdown(
                    choices=DBS.keys(),
                    label="Base de datos",
                    value=DBS.keys()[1]
                )

                query_input = gr.Textbox(
                    label="Ingresa tu query:",
                    value=""" 
                    / Ver tablas disponibles
                    SELECT name
                    FROM sqlite_master
                    WHERE type = 'table'
                     """,
                    
                    lines=50
                )

                boton = gr.Button("Run Query")

            # Lado derecho: resultado
            with gr.Column(scale=2):
                salida = gr.Dataframe(
                    label="Resultado",
                    wrap=True,# Ajusta texto en celdas
                    interactive=False# Hace que las columnas tengan ancho uniforme
                   )

        boton.click(fn=run_query, inputs=[DBS[db_select], query_input], outputs=salida)

    return demo



### Iniciar interfaz

In [ ]:
# --- Lanzar interfaz dentro del notebook ---
demo = make_interface()
demo.launch()  # inline=True → se muestra dentro del notebook